In [1]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_parquet("hf://datasets/agents-course/unit3-invitees/data/train-00000-of-00001.parquet")

In [3]:
df.head(5)

,name,relation,description,email
0,Ada Lovelace,best friend,Lady Ada Lovelace is my best friend. She is an...,ada.lovelace@example.com
1,Dr. Nikola Tesla,old friend from university days,Dr. Nikola Tesla is an old friend from your un...,nikola.tesla@gmail.com
2,Marie Curie,no relation,Marie Curie was a groundbreaking physicist and...,marie.curie@example.com


In [4]:
import datasets
from langchain_core.documents import Document

# Load the dataset
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

# Convert dataset entries into Document objects
docs = [
    Document(
        page_content="\n".join([
            f"Name: {guest['name']}",
            f"Relation: {guest['relation']}",
            f"Description: {guest['description']}",
            f"Email: {guest['email']}"
        ]),
        metadata={"name": guest["name"]}
    )
    for guest in guest_dataset
]

README.md:   0%|          | 0.00/371 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.32k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3 [00:00<?, ? examples/s]

In [25]:
docs

[Document(metadata={'name': 'Ada Lovelace'}, page_content="Name: Ada Lovelace\nRelation: best friend\nDescription: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.\nEmail: ada.lovelace@example.com"),
 Document(metadata={'name': 'Dr. Nikola Tesla'}, page_content="Name: Dr. Nikola Tesla\nRelation: old friend from university days\nDescription: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.\nEmail: nikola.tesla@gmail.com"),
 Document(metadata={'name': 'Marie Curie'}, page_content='Name: Marie Curie\nRelation: no relation\nDescription: Marie Curie was a groundbreaking physicist 

In [9]:
# Our "Raw Data"
import json
fruit_dataset = [
    {
     "type": "Apple", 
     "color": "Red", 
     "taste": "Sweet"
     },
    {
     "type": "Lemon", 
     "color": "Yellow", 
     "taste": "Sour"
     }
]

res = [
    {
     "content": f"this fruit is {fr['type']} and of colour {fr['color']}.",
     "id":f"{fr['type']}_01"
     } for fr in fruit_dataset
]
print(json.dumps(res,indent=2))

[
  {
    "content": "this fruit is Apple and of colour Red.",
    "id": "Apple_01"
  },
  {
    "content": "this fruit is Lemon and of colour Yellow.",
    "id": "Lemon_01"
  }
]


In [28]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.tools import Tool

bm25_retriever = BM25Retriever.from_documents(docs)

def extract_text(query: str) -> str:
    """Retrieves detailed information about gala guests based on their name or relation."""
    results = bm25_retriever.invoke(query)
    if results:
        print("----res",results)
        return "\n\n".join([doc.page_content for doc in results[:3]])
    else:
        return "No matching guest information found."

guest_info_tool = Tool(
    name="guest_info_retriever",
    func=extract_text,
    description="Retrieves detailed information about gala guests based on their name or relation."
)

In [29]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os

HUGGINGFACEHUB_API_TOKEN = os.getenv("HF_TOKEN")
# Generate the chat interface, including the tools
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
)

chat = ChatHuggingFace(llm=llm, verbose=True)
tools = [guest_info_tool]
chat_with_tools = chat.bind_tools(tools)

# Generate the AgentState and Agent graph
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {
        "messages": [chat_with_tools.invoke(state["messages"])],
    }

## The graph
builder = StateGraph(AgentState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message requires a tool, route to tools
    # Otherwise, provide a direct response
    tools_condition,
)
builder.add_edge("tools", "assistant")
alfred = builder.compile()

messages = [HumanMessage(content="tell me about guest Tesla")]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

----res [Document(metadata={'name': 'Dr. Nikola Tesla'}, page_content="Name: Dr. Nikola Tesla\nRelation: old friend from university days\nDescription: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.\nEmail: nikola.tesla@gmail.com"), Document(metadata={'name': 'Marie Curie'}, page_content='Name: Marie Curie\nRelation: no relation\nDescription: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.\nEmail: marie.curie@example.com'), Document(metadata={'name': 'Ada Lovelace'}, page_content="Name: Ada Lovelace\nRelation: best friend\nDescription: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer program

In [36]:
from smolagents import DuckDuckGoSearchTool

# Initialize the DuckDuckGo search tool
search_tool = DuckDuckGoSearchTool()

# Example usage
results = search_tool("Who's the current President of France?")
print(results)

## Search Results

[President of France - Wikipedia](https://en.wikipedia.org/wiki/President_of_France)
The president of France is the ex officio co-prince of Andorra, grand master of the Legion of Honour and of the National Order of Merit, and protector of the Institut de France in Paris.

[Welcome to the website of the French Presidency](https://www.elysee.fr/en/)
Find out all the news of the President Emmanuel Macron on the official website and discover our pages on the history of the French Republic.

[Presidents Of France Since 1959](https://www.worldatlas.com/articles/presidents-of-france-since-1959.html)
Other powers include promulgating laws, to dissolve parliament, to order use of nuclear weapons, to receive foreign ambassadors, among others. Louis-Napoleon Bonaparte is considered the first President of France .He is the current French President who was elected in 2012.

[France – EU country | European Union](https://european-union.europa.eu/principles-countries-history/eu-cou